In [1]:
# =============================================================
# SISTEMA DE VERIFICACION AUTOMATICA DE DOCUMENTOS ESCOLARES
# Pipeline de 5 barreras: Actas de Nacimiento y CURPs
# Referencia del pipeline: Bulatov et al. (2021). MIDV-2020. arXiv:2107.00396
# =============================================================

import warnings                  # libreria para controlar mensajes de advertencia
warnings.filterwarnings('ignore') # ocultamos advertencias para mantener la salida limpia

import pandas as pd              # pandas: maneja tablas de datos (como Excel pero en Python)
from pathlib import Path         # Path: construye rutas de archivos que funcionan en Windows/Mac/Linux

# BASE_DIR: ruta absoluta a la carpeta raiz del proyecto (MiDataset/)
# El notebook vive en: MiDataset/modelo/modelo_notebook/
# Con ../../ subimos 2 carpetas hasta llegar a MiDataset/
# .resolve() convierte la ruta relativa a ruta absoluta completa
BASE_DIR = Path('../../').resolve()

# Leemos el archivo manifiesto.csv que tiene la lista de todas las imagenes
# con su ruta y su clase (actas_nacimiento o curp)
df = pd.read_csv(BASE_DIR / 'manifiesto.csv')

# Mostramos las primeras 5 filas para verificar que cargo correctamente
df.head()

,nombre_archivo_original,id_documento,id_pagina,clase,ruta_imagen,ruta_pdf_original,ancho_pixeles,alto_pixeles,puntuacion_calidad,necesita_revision,es_aumentada,fecha_procesado,notas
0,ABREGO HERNANDEZ BRISA FERNANDA LOTE 17049 - A...,ACT_0001,ACT_0001_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ABREGO HE...,2472.0,3228.0,0.5728,True,False,2026-07-13 19:03,NaN
1,ACOSTA HERNANDEZ MIGUEL ANGEL LOTE 17055 - ACT...,ACT_0002,ACT_0002_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ACOSTA HE...,2472.0,3228.0,0.7435,False,False,2026-07-13 19:03,NaN
2,acta de nacimiento Wendy Veronica Marcos Cruz ...,ACT_0003,ACT_0003_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\acta de n...,2481.0,3508.0,0.5870,True,False,2026-07-13 19:03,NaN
3,Acta_de_Nacimiento_HERJ060111HHGRMSA9.pdf,ACT_0004,ACT_0004_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\Acta_de_N...,2550.0,3300.0,0.9486,False,False,2026-07-13 19:03,NaN
4,ActaNacimiento.pdf,ACT_0005,ACT_0005_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ActaNacim...,2550.0,3300.0,0.8376,False,False,2026-07-13 19:03,NaN


In [2]:
# Análisis exploratorio preliminar de la consistencia del dataset

print('Registros totales:', df.shape[0])  # Cardinalidad del conjunto de datos
print('Columnas:        ', df.shape[1])   # Dimensionalidad de metadatos
print()

# Validación de integridad: detección de valores nulos (NaN) en metadatos
print('Valores nulos por columna:')
print(df.isnull().sum())
print()

# Distribución estadística y balanceo de clases en la muestra
df['clase'].value_counts()

Registros totales: 7248
Columnas:         13

Valores nulos por columna:
nombre_archivo_original       0
id_documento                  0
id_pagina                     0
clase                         0
ruta_imagen                   0
ruta_pdf_original          6040
ancho_pixeles              6040
alto_pixeles               6040
puntuacion_calidad            0
necesita_revision             0
es_aumentada                  0
fecha_procesado               0
notas                      1208
dtype: int64



clase
actas_nacimiento    3624
curp                3624
Name: count, dtype: int64

In [3]:
# =============================================================
# BARRERA 1: Clasificación de tipo de documento mediante CNN
# Objetivo: Identificar la categoría de documento (Acta o CURP)
# Metodología: Transfer Learning sobre arquitectura MobileNetV3-Small
# =============================================================

# Librerías para modelado profundo y métricas estadísticas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ── PASO 1: PREPROCESAMIENTO Y PARTICIÓN DEL DATASET ──────────
# Mapeo numérico de etiquetas nominales
CLASES     = {'actas_nacimiento': 0, 'curp': 1, 'otros': 2}
CLASES_INV = {v: k for k, v in CLASES.items()}

# Filtrado de consistencia física de imágenes existentes en almacenamiento
df_v = df[df['clase'].isin(CLASES)].copy()
df_v = df_v[df_v['ruta_imagen'].apply(lambda r: (BASE_DIR / r).exists())]

# Extracción de vectores de características y etiquetas objetivo
X = [str(BASE_DIR / r) for r in df_v['ruta_imagen']]
Y = [CLASES[c] for c in df_v['clase']]

# Partición estratificada (80% entrenamiento, 20% prueba)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)
print(f'Total: {len(X)} imágenes | Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}')

# ── PASO 2: PIPELINE DE TRANSFORMACIONES Y DATA AUGMENTATION ──
# Transformaciones para conjunto de entrenamiento (Aumentación de datos y regularización fotométrica)
tf_train = transforms.Compose([
    transforms.Resize((224, 224)),           # Escalado estándar para MobileNetV3
    transforms.RandomHorizontalFlip(p=0.3),  # Regularización por simetría horizontal
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Variaciones controladas de exposición
    transforms.ToTensor(),                   # Conversión a tensor [0.0, 1.0]
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Normalización estadística de ImageNet
])

# Transformaciones para conjunto de validación/prueba (Métricas sin alteración fotométrica)
tf_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Clase personalizada para carga bajo demanda (Lazy Loading) de imágenes
class DocumentDataset(Dataset):
    def __init__(self, rutas, etiquetas, transform):
        self.rutas     = rutas
        self.etiquetas = etiquetas
        self.transform = transform

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert('RGB')
        imagen_lista = self.transform(img)
        etiqueta = torch.tensor(self.etiquetas[idx], dtype=torch.long)
        return imagen_lista, etiqueta

# Definición de cargadores de datos con procesamiento por lotes (Minibatch size = 8)
loader_train = DataLoader(DocumentDataset(X_train, Y_train, tf_train), batch_size=8, shuffle=True)
loader_test  = DataLoader(DocumentDataset(X_test, Y_test, tf_test),  batch_size=8, shuffle=False)
print(f'Lotes de entrenamiento: {len(loader_train)} | Lotes de prueba: {len(loader_test)}')

# ── PASO 3: ARQUITECTURA DE LA RED NEURONAL (TRANSFER LEARNING) ──
# Detección del acelerador de cómputo disponible
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de entrenamiento: {dispositivo}')

# Carga de arquitectura MobileNetV3-Small con pesos pre-entrenados en ImageNet
modelo = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Congelación de parámetros en el extractor de características (Fase 2)
for p in modelo.features.parameters():
    p.requires_grad = False

# Redefinición del clasificador final (Fase 3: Multi-Layer Perceptron personalizado)
modelo.classifier = nn.Sequential(
    nn.Linear(576, 256),  # Capa lineal reductora de dimensionalidad
    nn.Hardswish(),        # Función de activación no lineal optimizada
    nn.Dropout(p=0.3),    # Capa de regularización para mitigar overfitting (30%)
    nn.Linear(256, 3)  # 3 clases: actas_nacimiento, curp, otros     # Capa lineal de salida para dos clases lógicas
)

modelo = modelo.to(dispositivo)

# ── PASO 4: OPTIMIZACIÓN Y BUCLE DE ENTRENAMIENTO ────────────────
criterio = nn.CrossEntropyLoss() # Función de pérdida de entropía cruzada

# Optimizador Adam aplicado exclusivamente a parámetros activos (clasificador)
optimizador = optim.Adam(filter(lambda p: p.requires_grad, modelo.parameters()), lr=0.001)

# Scheduler para decrecimiento adaptativo del learning rate ante estancamiento
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='max', patience=5, factor=0.5)

mejor_prec  = 0.0
sin_mejora  = 0
mejor_pesos = None

print(f"\n{'Época':<8} {'Loss':<12} {'Precisión':<12} {'Estado'}")
print('-' * 45)

# Bucle iterativo de optimización de pesos y validación
for epoca in range(40):
    # Fase de optimización en conjunto de entrenamiento
    modelo.train()
    loss_total = 0.0
    for imgs, lbls in loader_train:
        imgs, lbls = imgs.to(dispositivo), lbls.to(dispositivo)
        optimizador.zero_grad()
        salida = modelo(imgs)
        loss = criterio(salida, lbls)
        loss.backward()  # Cálculo automático de gradientes
        optimizador.step() # Actualización de parámetros
        loss_total += loss.item()

    # Fase de validación determinística en conjunto de prueba
    modelo.eval()
    correctas, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader_test:
            salida = modelo(imgs.to(dispositivo))
            _, pred = torch.max(salida, 1)
            total     += lbls.size(0)
            correctas += (pred.cpu() == lbls).sum().item()

    prec = correctas / total
    scheduler.step(prec)

    # Preservación de la mejor configuración de pesos hallada
    if prec > mejor_prec:
        mejor_prec  = prec
        sin_mejora  = 0
        mejor_pesos = {k: v.clone() for k, v in modelo.state_dict().items()}
        estado = '<- mejor'
    else:
        sin_mejora += 1
        estado = ''

    print(f'{epoca+1:<8} {loss_total/len(loader_train):<12.4f} {prec*100:<11.1f}% {estado}')

    # Early Stopping: Detiene el entrenamiento si no se registra mejoría en 15 épocas
    if sin_mejora >= 15:
        print(f'Early stopping en época {epoca+1}: sin mejora por {sin_mejora} épocas')
        break

# Restauración del modelo con la mejor configuración de pesos
modelo.load_state_dict(mejor_pesos)
modelo.eval()
print(f'\nMejor precisión obtenida en conjunto de prueba: {mejor_prec*100:.1f}%')

# Evaluación de rendimiento mediante métricas estándar
preds_all, lbls_all = [], []
with torch.no_grad():
    for imgs, lbls in loader_test:
        _, pred = torch.max(modelo(imgs.to(dispositivo)), 1)
        preds_all.extend(pred.cpu().numpy())
        lbls_all.extend(lbls.numpy())
print(classification_report(lbls_all, preds_all, target_names=list(CLASES.keys())))

# Matriz de confusión para análisis de falsos positivos/negativos
pd.DataFrame(
    confusion_matrix(lbls_all, preds_all),
    index=[f'Real: {n}' for n in CLASES],
    columns=[f'Pred: {n}' for n in CLASES]
)
# ── Persistencia del modelo entrenado ─────────────────────────
# Serialización de los pesos del modelo en formato PyTorch (.pth)
# para habilitar inferencia posterior sin reentrenamiento.
# Ruta: MiDataset/modelo/modelo_barrera1.pth
ruta_guardado = BASE_DIR / 'modelo' / 'modelo_barrera1.pth'
torch.save(modelo.state_dict(), ruta_guardado)
print(f'Modelo serializado: {ruta_guardado}')
print(f'Tamanio: {ruta_guardado.stat().st_size / 1024:.1f} KB')

Total: 7248 imágenes | Entrenamiento: 5798 | Prueba: 1450
Lotes de entrenamiento: 725 | Lotes de prueba: 182
Dispositivo de entrenamiento: cpu

Época    Loss         Precisión    Estado
---------------------------------------------
1        0.0679       99.9       % <- mejor
2        0.0410       99.9       % 
3        0.0252       99.9       % 
4        0.0395       99.9       % 
5        0.0134       99.9       % 
6        0.0302       99.9       % 
7        0.0145       99.9       % 
8        0.0083       99.9       % 
9        0.0052       99.9       % 
10       0.0083       99.9       % 
11       0.0080       99.9       % 
12       0.0054       99.9       % 
13       0.0088       99.9       % 
14       0.0040       99.9       % 
15       0.0056       99.9       % 
16       0.0019       99.9       % 
Early stopping en época 16: sin mejora por 15 épocas

Mejor precisión obtenida en conjunto de prueba: 99.9%
                  precision    recall  f1-score   support

actas_nacimiento 

In [4]:
# ── PASO 5: EVALUACIÓN DE BARRERA 1 SOBRE EL DATASET COMPLETO ───
# Inferencia y filtro con umbral de decisión determinístico del 70%

def barrera_1(ruta):
    # Preparación de la imagen de entrada para inferencia
    tensor = tf_test(Image.open(ruta).convert('RGB')).unsqueeze(0).to(dispositivo)

    # Obtención de probabilidades lógicas de clase (probabilidades Softmax)
    with torch.no_grad():
        probs = torch.softmax(modelo(tensor), dim=1)[0]

    # Selección de la clase con mayor confianza
    idx  = int(probs.argmax())
    conf = float(probs.max())
    clase = CLASES_INV[idx]

    # Filtrado estricto por umbral de confianza mínimo
    if conf < 0.70:
        return False, clase, conf, (
            f'BARRERA 1 FALLIDA: tipo de documento no reconocido '
            f'({conf*100:.1f}% de confianza, mínimo requerido: 70%). '
            f'Sube una imagen más clara y bien encuadrada.'
        )
    return True, clase, conf, f"Documento reconocido como '{clase}' con {conf*100:.1f}% de confianza"

# Evaluación iterativa de todos los registros del dataset en Barrera 1
b1_res = []
for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    ok, clase_pred, conf, msg = barrera_1(ruta)
    b1_res.append({
        'archivo'    : fila.get('nombre_archivo_original', ruta.name),
        'clase_real' : fila['clase'],
        'clase_pred' : clase_pred,
        'confianza_%': round(conf*100, 1),
        'b1_ok'      : ok,
        'mensaje'    : msg
    })

B1 = pd.DataFrame(b1_res)

print('=== BARRERA 1 COMPLETADA ===')
print(f'Procesados: {len(B1)} archivos. Resultados listos para la Tabla Maestra.\n')


=== BARRERA 1 COMPLETADA ===
Procesados: 7248 archivos. Resultados listos para la Tabla Maestra.



In [ ]:
# =============================================================
# BARRERA 2 - VERIFICACIÓN DE CALIDAD DE IMAGEN
# =============================================================
# Pruebas: desenfoque (Laplaciano), ruido (varianza), rotación (Hough)
# Umbrales: blur>=80 | ruido>=10 | skew<=15°

import cv2
import numpy as np

BLUR_THRESHOLD  = 80.0
NOISE_THRESHOLD = 10.0
SKEW_THRESHOLD  = 15.0

def barrera_2(ruta_imagen):
    img = cv2.imread(str(ruta_imagen), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False, 'Error: imagen no legible.'

    blur = cv2.Laplacian(img, cv2.CV_64F).var()
    if blur < BLUR_THRESHOLD:
        return False, f'Error: imagen borrosa (nitidez={blur:.1f})'

    if np.var(img) < NOISE_THRESHOLD:
        return False, f'Error: contraste insuficiente'

    bordes = cv2.Canny(img, 50, 150, apertureSize=3)
    lineas = cv2.HoughLinesP(bordes,1,np.pi/180,threshold=100,minLineLength=100,maxLineGap=10)
    if lineas is not None:
        angulos = [np.degrees(np.arctan2(l[0][3]-l[0][1],l[0][2]-l[0][0]))
                   for l in lineas if l[0][2]!=l[0][0]]
        if angulos and abs(np.median(angulos)) > SKEW_THRESHOLD:
            return False, f'Error: rotación excesiva ({abs(np.median(angulos)):.1f}°)'

    return True, f'Calidad OK (blur={blur:.1f})'

b2_res = []
aprobados_b1 = set(B1[B1['b1_ok']]['archivo']) if 'B1' in globals() else set()

# Procesar en silencio (sin prints adentro del ciclo)
for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    if nombre not in aprobados_b1: continue
    ok, msg = barrera_2(ruta)
    b2_res.append({'archivo': nombre, 'b2_ok': ok, 'mensaje': msg})

B2 = pd.DataFrame(b2_res)
print('=== BARRERA 2 COMPLETADA ===')
print(f'Procesados: {len(B2)} archivos. Resultados listos para la Tabla Maestra.\n')



In [ ]:
# =============================================================
# BARRERA 3 - EXTRACCION DE INFORMACION CLAVE (KIE) Y OCR
# =============================================================
# Motor: EasyOCR ['es','en'] - soporte nativo espanol
# Preprocesamiento: CLAHE + Otsu (PreP-OCR, 2025)
# Validacion CURP: Regex patron RENAPO 18 chars
# Validacion Acta: 4 campos obligatorios con variantes OCR
# Correccion heuristica: O/0, I/1, S/5, B/8 (UNED, 2024)

import re
import cv2
import easyocr
import unicodedata
import pandas as pd

lector_ocr = easyocr.Reader(['es', 'en'], gpu=False)

def normalizar_texto_hispano(texto):
    # Quita acentos con NFD; protege la ñ con placeholder
    texto = texto.upper().replace('\u00d1', '||ENYE||')
    nfd = unicodedata.normalize('NFD', texto)
    limpio = ''.join([c for c in nfd if unicodedata.category(c) != 'Mn'])
    return limpio.replace('||ENYE||', '\u00d1')

def preprocesar_para_ocr(ruta):
    # CLAHE (8x8) + Otsu: mejora contraste antes del OCR
    img  = cv2.imread(str(ruta))
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    suavizado = cv2.GaussianBlur(clahe.apply(gris), (3, 3), 0)
    _, binaria = cv2.threshold(suavizado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binaria

def corregir_confusion_ocr(texto):
    # 3 variantes para corregir confusiones visuales del OCR
    variantes = [texto]
    variantes.append(texto.replace('0', 'O').replace('1', 'I').replace('5', 'S').replace('8', 'B'))
    variantes.append(texto.replace('O', '0').replace('I', '1').replace('S', '5'))
    return variantes

PATRON_CURP = re.compile(
    r'[A-Z]{4}[0-9]{6}[HM][A-Z]{2}[B-DF-HJ-NP-TV-Z]{3}[A-Z0-9][0-9]'
)

CAMPOS_ACTA = {
    'ACTA'      : ['ACTA', 'ACT4'],
    'NACIMIENTO': ['NACIMIENTO', 'NACI MIENTO'],
    'NOMBRE'    : ['NOMBRE', 'N0MBRE'],
    'MUNICIPIO' : ['MUNICIPIO', 'MPIO'],
}

def barrera_3(ruta, clase_documento):
    img_mejorada = preprocesar_para_ocr(ruta)
    resultados = lector_ocr.readtext(
        img_mejorada, detail=1, paragraph=False,
        contrast_ths=0.1, adjust_contrast=0.5
    )
    if not resultados:
        return False, 0.0, [], 'Error: No se detecto texto.'

    textos_ok = [(det[1], det[2]) for det in resultados if det[2] >= 0.30]
    if not textos_ok:
        return False, 0.0, [], 'Error: Confianza insuficiente.'

    confianza = sum(c for _, c in textos_ok) / len(textos_ok)
    texto = normalizar_texto_hispano(' '.join([t for t, _ in textos_ok]))

    if clase_documento == 'curp':
        curp = None
        for variante in corregir_confusion_ocr(texto):
            match = PATRON_CURP.search(variante)
            if match:
                curp = match.group(0)
                break
        if curp:
            return True, confianza, [curp], 'CURP Validado: ' + curp
        return False, confianza, [], 'Error: CURP no detectado.'

    elif clase_documento == 'actas_nacimiento':
        hallados = [c for c, v in CAMPOS_ACTA.items() if any(x in texto for x in v)]
        faltantes = [c for c in CAMPOS_ACTA if c not in hallados]
        if len(faltantes) >= 2:
            return False, confianza, hallados, 'Error: Faltan ' + str(faltantes)
        return True, confianza, hallados, 'Metadatos OK: ' + str(hallados)

    return False, 0.0, [], 'Clase no soportada.'

b3_res = []
aprobados_b2 = set(B2[B2['b2_ok']]['archivo']) if 'B2' in globals() else set()

for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    if nombre not in aprobados_b2: continue
    fila_b1 = B1[B1['archivo'] == nombre]
    if fila_b1.empty: continue
    clase_pred = fila_b1.iloc[0]['clase_pred']
    ok, conf, campos, msg = barrera_3(ruta, clase_pred)
    b3_res.append({
        'archivo': nombre, 'clase_pred': clase_pred,
        'confianza_ocr_%': round(conf*100, 1),
        'campos_hallados': str(campos),
        'b3_ok': ok, 'mensaje': msg
    })

B3 = pd.DataFrame(b3_res)

print('=== BARRERA 3 COMPLETADA ===')
print(f'Procesados: {len(B3)} archivos. Resultados listos para la Tabla Maestra.\n')


In [ ]:
# ==============================================================
# TABLA MAESTRA - RESULTADO FINAL DEL PIPELINE
# ==============================================================
# Consolida B1 + B2 + B3. Exporta resultados_pipeline.json.

import pandas as pd
import json

def construir_tabla_maestra(B1, B2, B3):
    # Evitar multiplicación de filas eliminando duplicados por nombre de archivo
    df_b1_raw = B1.drop_duplicates(subset=['archivo'])
    df_b2_raw = B2.drop_duplicates(subset=['archivo']) if not B2.empty else B2
    df_b3_raw = B3.drop_duplicates(subset=['archivo']) if not B3.empty else B3

    col_conf_b1 = next((c for c in ['confianza_b1_%','confianza_%'] if c in df_b1_raw.columns), None)
    cols_b1 = ['archivo','clase_pred','b1_ok','mensaje']
    if col_conf_b1: cols_b1.insert(2, col_conf_b1)
    df_b1 = df_b1_raw[cols_b1].copy()
    rename_b1 = {'mensaje':'obs_b1'}
    if col_conf_b1: rename_b1[col_conf_b1] = 'conf_b1_%'
    df_b1.rename(columns=rename_b1, inplace=True)
    if 'conf_b1_%' not in df_b1.columns: df_b1['conf_b1_%'] = 'N/D'

    if not df_b2_raw.empty:
        col_msg_b2 = 'mensaje' if 'mensaje' in df_b2_raw.columns else df_b2_raw.columns[-1]
        df_b2 = df_b2_raw[['archivo','b2_ok',col_msg_b2]].copy()
        df_b2.rename(columns={col_msg_b2:'obs_b2'}, inplace=True)
    else:
        df_b2 = pd.DataFrame(columns=['archivo', 'b2_ok', 'obs_b2'])

    if not df_b3_raw.empty:
        col_msg_b3  = 'mensaje' if 'mensaje' in df_b3_raw.columns else df_b3_raw.columns[-1]
        col_conf_b3 = next((c for c in ['confianza_ocr_%','confianza_%'] if c in df_b3_raw.columns), None)
        cols_b3 = ['archivo','b3_ok',col_msg_b3]
        if col_conf_b3: cols_b3.insert(2, col_conf_b3)
        df_b3 = df_b3_raw[cols_b3].copy()
        rename_b3 = {col_msg_b3:'obs_b3'}
        if col_conf_b3: rename_b3[col_conf_b3] = 'conf_ocr_%'
        df_b3.rename(columns=rename_b3, inplace=True)
        if 'conf_ocr_%' not in df_b3.columns: df_b3['conf_ocr_%'] = 'N/D'
    else:
        df_b3 = pd.DataFrame(columns=['archivo', 'b3_ok', 'obs_b3', 'conf_ocr_%'])

    master = df_b1.merge(df_b2, on='archivo', how='left').merge(df_b3, on='archivo', how='left')
    for col, val in [('b2_ok','N/A'),('b3_ok','N/A'),('obs_b2','No procesado'),
                     ('obs_b3','No procesado'),('conf_ocr_%','N/D')]:
        master[col] = master[col].fillna(val)
    return master

def calcular_estado_final(row):
    return 'ACEPTADO' if row['b1_ok']==True and row['b2_ok']==True and row['b3_ok']==True else 'RECHAZADO'

def construir_observaciones(row):
    obs = ['B1: ' + str(row['obs_b1'])]
    if 'No procesado' not in str(row['obs_b2']): obs.append('B2: ' + str(row['obs_b2']))
    if 'No procesado' not in str(row['obs_b3']): obs.append('B3: ' + str(row['obs_b3']))
    return ' | '.join(obs)

if 'B1' in globals() and 'B2' in globals() and 'B3' in globals():
    df_master = construir_tabla_maestra(B1, B2, B3)
    df_master['estado_final']  = df_master.apply(calcular_estado_final, axis=1)
    df_master['observaciones'] = df_master.apply(construir_observaciones, axis=1)

    cols = ['archivo','clase_pred','conf_b1_%','b1_ok','b2_ok','b3_ok','conf_ocr_%','estado_final']
    print(df_master[cols].to_string(index=False))
    print('\n' + df_master[['archivo','observaciones']].to_string(index=False))

    total = len(df_master)
    aceptados = (df_master['estado_final']=='ACEPTADO').sum()
    print('Total: '+str(total)+' | ACEPTADOS: '+str(aceptados)+' | RECHAZADOS: '+str(total-aceptados))

    ruta_json = BASE_DIR / 'resultados_pipeline.json'
    with open(ruta_json, 'w', encoding='utf-8') as f:
        json.dump({'resumen':{'total':total,'aceptados':int(aceptados),'rechazados':int(total-aceptados)},
                   'documentos':df_master.to_dict(orient='records')}, f, ensure_ascii=False, indent=2)
    print('JSON exportado: ' + str(ruta_json))
else:
    print('[INFO] Ejecuta primero B1, B2 y B3.')

